# KOA Semantic Unpacking

Este notebook lê um arquivo TXT (texto livre) e usa Gemini para extrair uma representação semântica canônica em JSON (com evidências). Em seguida, normaliza (IDs estáveis, deduplicação leve) e exporta:

- `KOA_semantic_requirements_normalized.yml` (semantic requirements normalizado para o framework KOA)
- `KOA_semantic_requirements_normalized.pl` (opcional) fatos Prolog para grounding e integração futura

> Observação: o código não depende do vocabulário do TXT.  
Ele depende apenas de um schema canônico fixo e de validação determinística.


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [2]:
!pip uninstall google-generativeai -y
!pip install google-genai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6


In [3]:
!pip -q install pyyaml jsonschema

In [4]:
# Importa bibliotecas padrão e do Gemini
import os
import re
import json
import time
import datetime

from typing import Any, Dict, List, Optional, Tuple
from jsonschema import validate, Draft202012Validator
import yaml
import hashlib
import math

from google import genai

In [5]:
# Recupera chave Gemini com secrets do Colab
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [6]:
# Configuração de pastas e arquivos
directory = '/content/drive/My Drive/KOA/onboarding/modelos/'
persist_directory = '/content/drive/My Drive/KOA/onboarding/modelos/'

txt_file_path = os.path.join(directory, "KOA_user_information_needs_gseg.txt")

output_json_path  = os.path.join(persist_directory, "KOA_semantic_requirements_extracted.json")
output_yaml_path  = os.path.join(persist_directory, "KOA_semantic_requirements_normalized.yml")
output_prolog_path = os.path.join(persist_directory, "KOA_semantic_requirements_normalized.pl")

organization = "BNDES"   # opcional
domain_name = "security_risks"
language_code = "pt"

In [7]:
# Lê arquivo de necessidades de informação do usuário
def read_txt_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

raw_text = read_txt_file(txt_file_path)
print("TXT chars:", len(raw_text))
print(raw_text[:500])


TXT chars: 8944
Dentro da governança de riscos do BNDES, risco operacional é um conceito mais amplo que se refere à possibilidade de perdas decorrentes de falhas, inadequações ou eventos externos que afetem processos, pessoas, sistemas ou controles internos.
Definição de Risco à Segurança da Informação: potencial de violação da integridade, confidencialidade, disponibilidade ou autenticidade da informação de propriedade ou custodiada pelo Sistema BNDES em decorrência da exploração de uma ou mais vulnerabilidade


In [8]:
# Funções de apoio
def chunk_text(text: str, max_chars: int = 6000, overlap: int = 300) -> List[str]:
    """Divide texto em blocos ~max_chars, com sobreposição leve.
    Independente de vocabulário: usa apenas tamanho e quebras de parágrafo.
    """
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    chunks = []
    buf = ""
    for p in paras:
        if len(buf) + len(p) + 2 <= max_chars:
            buf = (buf + "\n\n" + p).strip()
        else:
            if buf:
                chunks.append(buf)
            # se o parágrafo for gigante, quebra bruto
            if len(p) > max_chars:
                for i in range(0, len(p), max_chars - overlap):
                    chunks.append(p[i:i + max_chars])
                buf = ""
            else:
                buf = p
    if buf:
        chunks.append(buf)

    # adiciona overlap textual simples (mantém última parte do chunk anterior)
    if overlap > 0 and len(chunks) > 1:
        out = []
        prev_tail = ""
        for ch in chunks:
            merged = (prev_tail + "\n\n" + ch).strip() if prev_tail else ch
            out.append(merged)
            prev_tail = ch[-overlap:]
        chunks = out

    return chunks

chunks = chunk_text(raw_text)
print("Chunks:", len(chunks))
print("Chunk[0] chars:", len(chunks[0]))


Chunks: 2
Chunk[0] chars: 5746


In [9]:
# Define schema canônico. Mantém o pipeline independente do vocabulário do TXT.
SCHEMA_CANONICAL = {
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "type": "object",
  "required": ["doc_meta", "concepts", "entities", "relations", "constraints", "taxonomies", "decision_rules", "question_frames"],
  "properties": {
    "doc_meta": {
      "type": "object",
      "required": ["language", "source_id"],
      "properties": {
        "language": {"type": "string"},
        "domain_hint": {"type": ["string", "null"]},
        "source_id": {"type": "string"}
      },
      "additionalProperties": True
    },
    "concepts": {"type": "array", "items": {"$ref": "#/$defs/concept"}},
    "entities": {"type": "array", "items": {"$ref": "#/$defs/entity"}},
    "relations": {"type": "array", "items": {"$ref": "#/$defs/relation"}},
    "constraints": {"type": "array", "items": {"$ref": "#/$defs/constraint"}},
    "taxonomies": {"type": "array", "items": {"$ref": "#/$defs/taxonomy"}},
    "decision_rules": {"type": "array", "items": {"$ref": "#/$defs/decision_rule"}},
    "question_frames": {"type": "array", "items": {"$ref": "#/$defs/question_frame"}}
  },
  "$defs": {
    "evidence": {
      "type": "object",
      "required": ["quote", "loc"],
      "properties": {
        "quote": {"type": "string"},
        "loc": {"type": "string"}
      },
      "additionalProperties": True
    },
    "concept": {
      "type": "object",
      "required": ["id", "label", "definition", "aliases", "evidence"],
      "properties": {
        "id": {"type": "string"},
        "label": {"type": "string"},
        "definition": {"type": "string"},
        "aliases": {"type": "array", "items": {"type": "string"}},
        "evidence": {"type": "array", "items": {"$ref": "#/$defs/evidence"}, "minItems": 1}
      },
      "additionalProperties": True
    },
    "entity": {
      "type": "object",
      "required": ["id", "label", "entity_type", "aliases", "evidence"],
      "properties": {
        "id": {"type": "string"},
        "label": {"type": "string"},
        "entity_type": {"type": "string"},
        "aliases": {"type": "array", "items": {"type": "string"}},
        "evidence": {"type": "array", "items": {"$ref": "#/$defs/evidence"}, "minItems": 1}
      },
      "additionalProperties": True
    },
    "relation": {
      "type": "object",
      "required": ["id", "predicate", "arguments", "polarity", "evidence"],
      "properties": {
        "id": {"type": "string"},
        "predicate": {"type": "string"},
        "arguments": {
          "type": "array",
          "items": {
            "type": "object",
            "required": ["role", "ref"],
            "properties": {
              "role": {"type": "string"},
              "ref": {"type": "string"}
            },
            "additionalProperties": True
          },
          "minItems": 2
        },
        "polarity": {"type": "string", "enum": ["asserted", "possible", "forbidden"]},
        "evidence": {"type": "array", "items": {"$ref": "#/$defs/evidence"}, "minItems": 1}
      },
      "additionalProperties": True
    },
    "constraint": {
      "type": "object",
      "required": ["id", "kind", "expression", "evidence"],
      "properties": {
        "id": {"type": "string"},
        "kind": {"type": "string", "enum": ["scope", "precondition", "filter", "requirement", "exception", "other"]},
        "expression": {"type": "object"},
        "evidence": {"type": "array", "items": {"$ref": "#/$defs/evidence"}, "minItems": 1}
      },
      "additionalProperties": True
    },
    "taxonomy": {
      "type": "object",
      "required": ["id", "label", "taxonomy_type", "levels", "evidence"],
      "properties": {
        "id": {"type": "string"},
        "label": {"type": "string"},
        "taxonomy_type": {"type": "string", "enum": ["ordinal", "nominal", "boolean"]},
        "levels": {
          "type": "array",
          "items": {
            "type": "object",
            "required": ["id", "label", "order", "description"],
            "properties": {
              "id": {"type": "string"},
              "label": {"type": "string"},
              "order": {"type": "integer"},
              "description": {"type": "string"},
              "prolog_atom": {"type": ["string", "null"]}
            },
            "additionalProperties": True
          },
          "minItems": 1
        },
        "evidence": {"type": "array", "items": {"$ref": "#/$defs/evidence"}, "minItems": 1}
      },
      "additionalProperties": True
    },
    "decision_rule": {
      "type": "object",
      "required": ["id", "label", "rules", "evidence"],
      "properties": {
        "id": {"type": "string"},
        "label": {"type": "string"},
        "rules": {
          "type": "array",
          "items": {
            "type": "object",
            "required": ["if", "then"],
            "properties": {
              "if": {"type": "object"},
              "then": {"type": "object"}
            },
            "additionalProperties": True
          }
        },
        "evidence": {"type": "array", "items": {"$ref": "#/$defs/evidence"}, "minItems": 1}
      },
      "additionalProperties": True
    },
    "question_frame": {
      "type": "object",
      "required": ["id", "question", "intent", "targets", "slots", "evidence"],
      "properties": {
        "id": {"type": "string"},
        "question": {"type": "string"},
        "intent": {"type": "string"},
        "targets": {"type": "array", "items": {"type": "string"}},
        "slots": {
          "type": "array",
          "items": {
            "type": "object",
            "required": ["name", "slot_type", "required", "ask_if_missing"],
            "properties": {
              "name": {"type": "string"},
              "slot_type": {"type": "string"},
              "required": {"type": "boolean"},
              "ask_if_missing": {"type": "string"}
            },
            "additionalProperties": True
          }
        },
        "evidence": {"type": "array", "items": {"$ref": "#/$defs/evidence"}, "minItems": 1}
      },
      "additionalProperties": True
    }
  },
  "additionalProperties": True
}

_validator = Draft202012Validator(SCHEMA_CANONICAL)

def validate_canonical(obj: Dict[str, Any]) -> List[str]:
    errors = []
    for e in sorted(_validator.iter_errors(obj), key=lambda x: x.path):
        errors.append(f"{list(e.path)}: {e.message}")
    return errors

In [10]:
# Define o prompt para o Gemini
EXTRACTION_INSTRUCTIONS = """Você é um extrator semântico para o framework KnowOntoAsk.
Receberá um TEXTO LIVRE (em língua natural) e deve produzir APENAS um JSON estritamente válido,
conforme o schema canônico fornecido.

Regras:
1) NÃO invente informação: cada item extraído deve conter evidence.quote com um trecho que exista no texto.
2) Se o texto contiver definições, extraia em concepts.
3) Se contiver listas/níveis/escala/taxonomia (mesmo sem nome), extraia em taxonomies (taxonomy_type adequado).
4) Se contiver regras de decisão, matrizes ou condições -> extraia em decision_rules.
5) Se contiver perguntas (explícitas) ou requisitos em forma de pergunta -> extraia em question_frames.
6) relations devem usar predicate em snake_case e arguments com role/ref.
7) Se algum bloco não contiver um tipo, retorne a lista correspondente vazia (não remova chaves do JSON).

Formato de loc:
- use algo como "chunk:N" e, se possível, "chunk:N:excerpt".
""".strip()

def build_prompt(text_chunk: str, chunk_idx: int) -> str:
    # Schema como JSON compacto ajuda a reduzir tokens, mas ainda orienta a estrutura
    schema_compact = json.dumps(SCHEMA_CANONICAL, ensure_ascii=False)
    return (
        f"INSTRUCTIONS:\n{EXTRACTION_INSTRUCTIONS}\n\n"
        f"SCHEMA_CANONICAL_JSON:\n{schema_compact}\n\n"
        f"TEXT (chunk:{chunk_idx}):\n\"\"\"\n{text_chunk}\n\"\"\"\n"
        f"\nOUTPUT: (return ONLY valid JSON)"
    )


In [11]:
# Função de chamada ao Gemini
def call_gemini_json(prompt: str, model_name: str = "gemini-2.0-flash-lite",
                     sleep_seconds: int = 20, max_tries: int = 6) -> Dict[str, Any]:
    """
    Chama Gemini e retorna dict parseado.
    Robusto a:
      - ```json fences
      - caracteres de controle dentro de strings (strict=False)
      - BOM e invisíveis comuns
    """

    client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'))

    def _strip_fences(s: str) -> str:
        s = s.strip()
        s = re.sub(r"^\s*```(?:json)?\s*", "", s, flags=re.IGNORECASE).strip()
        s = re.sub(r"\s*```\s*$", "", s).strip()
        return s

    def _remove_bad_controls(s: str) -> str:
        # remove BOM
        s = s.replace("\ufeff", "")
        # remove controles ASCII 0x00-0x1F, exceto \n \r \t
        s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", s)
        return s

    last_err = None
    for attempt in range(1, max_tries + 1):
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=prompt
            )
            text = resp.text

            if isinstance(text, dict):
                return text
            if not isinstance(text, str):
                raise TypeError(f"Gemini returned non-str: {type(text)}")

            t = _strip_fences(text)
            t = _remove_bad_controls(t)

            # parse tolerante a controle dentro de strings
            obj = json.loads(t, strict=False)
            return obj

        except Exception as e:
            last_err = e
            print(f"⚠️ attempt {attempt}/{max_tries} failed: {e}")
            time.sleep(sleep_seconds)

    raise RuntimeError(f"Gemini failed after {max_tries} tries. Last error: {last_err}")


In [12]:
# Funções de apoio ao merge das saídas
def empty_canonical(language: str = "pt", source_id: str = "free_text") -> Dict[str, Any]:
    return {
        "doc_meta": {"language": language, "domain_hint": None, "source_id": source_id},
        "concepts": [],
        "entities": [],
        "relations": [],
        "constraints": [],
        "taxonomies": [],
        "decision_rules": [],
        "question_frames": []
    }

def merge_canonical(dst: Dict[str, Any], src: Dict[str, Any]) -> Dict[str, Any]:
    # concatena listas. Deduplicação/IDs estáveis vem depois.
    for k in ["concepts","entities","relations","constraints","taxonomies","decision_rules","question_frames"]:
        dst[k].extend(src.get(k, []))
    return dst

canonical_all = empty_canonical(language="pt", source_id=os.path.basename(txt_file_path))

for i, ch in enumerate(chunks):
    prompt = build_prompt(ch, i)
    obj = call_gemini_json(prompt)

    errs = validate_canonical(obj)
    if errs:
        # tentativa de correção: pede para o LLM consertar mantendo evidências
        fix_prompt = (
            prompt
            + "\n\nVALIDATION_ERRORS:\n"
            + "\n".join(errs[:30])
            + "\n\nCorrija o JSON para validar no schema. Não invente novos itens; preserve evidências."
        )
        obj = call_gemini_json(fix_prompt)
        errs2 = validate_canonical(obj)
        if errs2:
            raise RuntimeError("JSON ainda inválido após correção. Erros (parcial):\n" + "\n".join(errs2[:30]))

    canonical_all = merge_canonical(canonical_all, obj)

print("Extracted counts:", {k: len(canonical_all[k]) for k in canonical_all if isinstance(canonical_all[k], list)})


Extracted counts: {'concepts': 7, 'entities': 0, 'relations': 0, 'constraints': 0, 'taxonomies': 3, 'decision_rules': 1, 'question_frames': 11}


In [13]:
# Funções de normalização (IDs estáveis + deduplicação leve)
def stable_id(prefix: str, s: str, n: int = 10) -> str:
    h = hashlib.sha1(s.encode("utf-8")).hexdigest()[:n]
    return f"{prefix}{h}"

def normalize_items_by_label(items: List[Dict[str, Any]], prefix: str, label_key: str) -> List[Dict[str, Any]]:
    seen = {}
    out = []
    for it in items:
        label = (it.get(label_key) or "").strip()
        signature = label.lower()
        if not signature:
            # fallback para evidência
            ev = it.get("evidence",[{}])[0].get("quote","")
            signature = ev.lower().strip()
        if not signature:
            signature = json.dumps(it, ensure_ascii=False)

        if signature in seen:
            # mescla aliases/evidência
            tgt = seen[signature]
            # aliases
            if "aliases" in it and "aliases" in tgt:
                tgt["aliases"] = sorted(set(tgt["aliases"]) | set(it["aliases"]))
            # evidências
            tgt["evidence"] = tgt.get("evidence", []) + it.get("evidence", [])
            continue

        it2 = dict(it)
        it2["id"] = stable_id(prefix, signature)
        seen[signature] = it2
        out.append(it2)
    return out

def normalize_predicates(relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for r in relations:
        r2 = dict(r)
        pred = (r2.get("predicate") or "").strip()
        pred = re.sub(r"[^a-zA-Z0-9_]+", "_", pred).lower()
        pred = re.sub(r"_+", "_", pred).strip("_")
        r2["predicate"] = pred or "related_to"
        out.append(r2)
    return out

canonical_norm = dict(canonical_all)
canonical_norm["concepts"] = normalize_items_by_label(canonical_all["concepts"], "C_", "label")
canonical_norm["entities"] = normalize_items_by_label(canonical_all["entities"], "E_", "label")
canonical_norm["taxonomies"] = normalize_items_by_label(canonical_all["taxonomies"], "T_", "label")
canonical_norm["question_frames"] = normalize_items_by_label(canonical_all["question_frames"], "QF_", "question")

canonical_norm["relations"] = normalize_predicates(canonical_all["relations"])
canonical_norm["relations"] = normalize_items_by_label(canonical_norm["relations"], "R_", "predicate")

canonical_norm["constraints"] = normalize_items_by_label(canonical_all["constraints"], "K_", "kind")
canonical_norm["decision_rules"] = normalize_items_by_label(canonical_all["decision_rules"], "D_", "label")

errs = validate_canonical(canonical_norm)
print("Normalized validation errors:", len(errs))
if errs:
    print("\n".join(errs[:20]))


Normalized validation errors: 0


In [14]:
# Exporta JSON
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(canonical_norm, f, ensure_ascii=False, indent=2)

print("Saved:", output_json_path)


Saved: /content/drive/My Drive/KOA/onboarding/modelos/KOA_semantic_requirements_extracted.json


In [15]:
# Função para construir YAML (semantic requirements normalizado)
def build_koa_yaml(canon: Dict[str, Any]) -> Dict[str, Any]:
    # Aqui fazemos um mapeamento simples:
    # - concepts -> concepts
    # - taxonomies -> categories
    # - question_frames -> question_frames
    # Mantemos também relations/constraints/decision_rules (útil para grounding e evolução).
    return {
        "schema_version": "koa-semreq-llm-0.3",
        "doc_meta": canon["doc_meta"],
        "concepts": [
            {
                "id": c["id"],
                "type": "concept",
                "label": c.get("label"),
                "definition": c.get("definition"),
                "aliases": c.get("aliases", []),
                "evidence": c.get("evidence", [])
            } for c in canon.get("concepts", [])
        ],
        "categories": [
            {
                "id": t["id"],
                "type": "category",
                "label": t.get("label"),
                "taxonomy_type": t.get("taxonomy_type"),
                "values": t.get("levels", []),
                "evidence": t.get("evidence", [])
            } for t in canon.get("taxonomies", [])
        ],
        "question_frames": [
            {
                "id": q["id"],
                "type": "question_frame",
                "question": q.get("question"),
                "intent": q.get("intent"),
                "targets": q.get("targets", []),
                "slots": q.get("slots", []),
                "evidence": q.get("evidence", []),

                # placeholders para integração Prolog (você preencherá ao alinhar com a KB):
                "prolog_mapping": q.get("prolog_mapping", None)
            } for q in canon.get("question_frames", [])
        ],
        "relations": canon.get("relations", []),
        "constraints": canon.get("constraints", []),
        "decision_rules": canon.get("decision_rules", [])
    }

koa_yaml = build_koa_yaml(canonical_norm)

with open(output_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(koa_yaml, f, allow_unicode=True, sort_keys=False, width=120)

print("Saved:", output_yaml_path)
print("Counts:",
      "concepts", len(koa_yaml["concepts"]),
      "categories", len(koa_yaml["categories"]),
      "question_frames", len(koa_yaml["question_frames"]))


Saved: /content/drive/My Drive/KOA/onboarding/modelos/KOA_semantic_requirements_normalized.yml
Counts: concepts 7 categories 3 question_frames 11


In [16]:
# Função para construir Prolog (opcional)
def prolog_escape(s: str) -> str:
    s = s.replace('\\', '\\\\').replace('"', '\\"')
    s = s.replace('\n', '\\n')
    return f'"{s}"'

def emit_prolog(koa: Dict[str, Any]) -> str:
    lines = []
    meta = koa.get("doc_meta", {})
    lines.append(f"% KOA Semantic Requirements (generated) - {datetime.datetime.utcnow().isoformat()}Z")
    lines.append(f"% source_id: {meta.get('source_id')}")
    lines.append("")

    # concepts
    for c in koa.get("concepts", []):
        cid = c["id"].lower()
        lines.append(f"concept({cid}).")
        if c.get("label"):
            lines.append(f"concept_label({cid}, {prolog_escape(c['label'])}).")
        if c.get("definition"):
            lines.append(f"concept_definition({cid}, {prolog_escape(c['definition'])}).")
        for a in c.get("aliases", []):
            lines.append(f"concept_alias({cid}, {prolog_escape(a)}).")
        for ev in c.get("evidence", []):
            lines.append(f"evidence({cid}, {prolog_escape(ev.get('quote',''))}, {prolog_escape(ev.get('loc',''))}).")
        lines.append("")

    # categories / taxonomies
    for cat in koa.get("categories", []):
        tid = cat["id"].lower()
        lines.append(f"category({tid}).")
        if cat.get("label"):
            lines.append(f"category_label({tid}, {prolog_escape(cat['label'])}).")
        tax_type = cat.get("taxonomy_type")
        if tax_type:
            lines.append(f"category_type({tid}, {tax_type}).")
        for lv in cat.get("values", []):
            lid = (lv.get("id") or stable_id('L_', json.dumps(lv, ensure_ascii=False))).lower()
            label = lv.get("label","")
            order = int(lv.get("order", 0))
            desc = lv.get("description","")
            lines.append(f"category_level({tid}, {lid}, {order}, {prolog_escape(label)}, {prolog_escape(desc)}).")
        for ev in cat.get("evidence", []):
            lines.append(f"evidence({tid}, {prolog_escape(ev.get('quote',''))}, {prolog_escape(ev.get('loc',''))}).")
        lines.append("")

    # question frames
    for q in koa.get("question_frames", []):
        qid = q["id"].lower()
        lines.append(f"question_frame({qid}).")
        if q.get("question"):
            lines.append(f"frame_question({qid}, {prolog_escape(q['question'])}).")
        if q.get("intent"):
            lines.append(f"frame_intent({qid}, {prolog_escape(q['intent'])}).")
        for t in q.get("targets", []):
            lines.append(f"frame_target({qid}, {prolog_escape(t)}).")
        for s in q.get("slots", []):
            nm = (s.get("name") or "slot").lower()
            st = s.get("slot_type","any")
            req = "true" if s.get("required") else "false"
            ask = s.get("ask_if_missing","")
            lines.append(f"frame_slot({qid}, {prolog_escape(nm)}, {prolog_escape(st)}, {req}, {prolog_escape(ask)}).")
        for ev in q.get("evidence", []):
            lines.append(f"evidence({qid}, {prolog_escape(ev.get('quote',''))}, {prolog_escape(ev.get('loc',''))}).")
        lines.append("")

    # relations (como ontologia auxiliar)
    for r in koa.get("relations", []):
        rid = r["id"].lower()
        pred = r.get("predicate","related_to")
        pol = r.get("polarity","possible")
        lines.append(f"relation({rid}, {prolog_escape(pred)}, {prolog_escape(pol)}).")
        for a in r.get("arguments", []):
            role = a.get("role","arg")
            ref = a.get("ref","")
            lines.append(f"relation_arg({rid}, {prolog_escape(role)}, {prolog_escape(ref)}).")
        for ev in r.get("evidence", []):
            lines.append(f"evidence({rid}, {prolog_escape(ev.get('quote',''))}, {prolog_escape(ev.get('loc',''))}).")
        lines.append("")

    return "\n".join(lines)

pl_text = emit_prolog(koa_yaml)
with open(output_prolog_path, "w", encoding="utf-8") as f:
    f.write(pl_text)

print("Saved:", output_prolog_path)


/tmp/ipython-input-935396920.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  lines.append(f"% KOA Semantic Requirements (generated) - {datetime.datetime.utcnow().isoformat()}Z")


Saved: /content/drive/My Drive/KOA/onboarding/modelos/KOA_semantic_requirements_normalized.pl


## Preview rápido

In [17]:
print("YAML preview (top-level keys):", list(koa_yaml.keys()))
print("First concept:", (koa_yaml["concepts"][:1] or [None])[0])
print("First category:", (koa_yaml["categories"][:1] or [None])[0])
print("First question frame:", (koa_yaml["question_frames"][:1] or [None])[0])


YAML preview (top-level keys): ['schema_version', 'doc_meta', 'concepts', 'categories', 'question_frames', 'relations', 'constraints', 'decision_rules']
First concept: {'id': 'C_7fbcda231b', 'type': 'concept', 'label': 'Risco operacional', 'definition': 'possibilidade de perdas decorrentes de falhas, inadequações ou eventos externos que afetem processos, pessoas, sistemas ou controles internos', 'aliases': [], 'evidence': [{'quote': 'risco operacional é um conceito mais amplo que se refere à possibilidade de perdas decorrentes de falhas, inadequações ou eventos externos que afetem processos, pessoas, sistemas ou controles internos', 'loc': 'chunk:0'}]}
First category: {'id': 'T_fb60c6d255', 'type': 'category', 'label': 'Contratos de terceirização', 'taxonomy_type': 'nominal', 'values': [{'id': 'l1', 'label': 'Alocação de mão-de-obra dedicada', 'order': 1, 'description': 'Atuam em postos de trabalho definidos no contrato', 'prolog_atom': None}, {'id': 'l2', 'label': 'Prestação de serviç